# Laboratorio 7

## Integrantes

| Nombre                            | Carnet | Usuario Git |
| --------------------------------- | ------ | ----------- |
| Edwin Jose Gabriel De Leon Garcia | 22809  | EJGDLG      |
| Gustavo Adolfo Cruz Bardales      | 22779  | G2309       |
| Josué Emanuel Say Garcia          | 22801  | JosueSay    |
| Mathew Alexander Cordero Aquino   | 22982  | donmatthiuz |


## Repositorio

[Link al Repositorio](https://github.com/donmatthiuz/RL/tree/lab7)

## Caso

Una empresa de ciberseguridad opera un sistema de detección de intrusiones en redes corporativas. El sistema monitorea el tráfico de red en tiempo real y debe decidir qué acción tomar ante comportamientos sospechosos: ignorar, alertar, bloquear temporalmente, o aislar el segmento afectado. El estado del sistema en cada instante es un vector de alta dimensión que incluye métricas de tráfico, patrones de comportamiento de usuarios, y logs de eventos recientes. La empresa quiere explorar si Deep RL puede mejorar la toma de decisiones frente a los sistemas basados en reglas que usan actualmente.

## Task 1

### Task 1.1

El equipo de ingeniería propone usar DQN estándar para este sistema. Antes de implementar cualquier cosa, usted debe hacer un análisis crítico de esa propuesta.

**a.** El estado del sistema es un vector de 256 dimensiones con valores continuos. Argumenten si DQN es arquitecturalmente apropiado para este tipo de estado. ¿Qué tipo de red usarían como aproximador de $Q(s,a;w)$ y por qué, considerando que el estado no es una imagen sino un vector numérico?

**Respuesta:** DQN es arquitecturalmente apropiado porque reemplaza la tabla Q de Q-learning por una red neuronal que aproxima $Q(s,a;w)$. Una tabla Q no es viable: un estado con 256 variables continuas produce un espacio de estados demasiado grande e imposible de enumerar. Se usaría una red feedforward o MLP totalmente conectada, con 256 entradas y cuatro salidas, una por acción. Esta arquitectura permite aprender relaciones entre las métricas, patrones de usuarios y logs del vector. No se requiere una CNN porque el estado no tiene estructura espacial como una imagen. Tampoco se requiere una red recurrente si el vector incorpora suficiente historial reciente para representar aproximadamente el estado de Markov.

**b.** El espacio de acciones tiene 4 opciones discretas. Argumenten si DQN o alguna de sus variantes es apropiado para este espacio de acciones, o si es necesario un algoritmo diferente.

**Respuesta:** El espacio de acciones es discreto y finito: ignorar, alertar, bloquear temporalmente y aislar el segmento. Por ello DQN es apropiado, ya que puede producir un valor Q por cada acción y el máximo del target se calcula sobre solo cuatro opciones. No es necesario usar algoritmos para control continuo, como DDPG o SAC. Sin embargo, Double DQN sería una mejora recomendable frente a DQN estándar, porque reduce la sobreestimación al separar la selección de la mejor acción con la red principal y su evaluación con la red objetivo.

**c.** En el dominio de detección de intrusiones, los eventos críticos (intrusiones reales) son extremadamente raros: ocurren en menos del 0.1% de los pasos de tiempo. Argumenten formalmente qué consecuencia tiene esa rareza sobre el buffer de Experience Replay y sobre la distribución de muestreo uniforme. ¿Qué variante de DQN resolvería este problema específicamente y cómo?

**Respuesta:** Sea $p=0.001$ la proporción máxima de intrusiones reales en el buffer. Bajo muestreo uniforme, una transición crítica tiene probabilidad $p$ de ser seleccionada y un minibatch de tamaño $B$ contiene en promedio $Bp$ intrusiones. Por ejemplo, para $B=32$, el valor esperado es $32(0.001)=0.032$ intrusiones por minibatch y la probabilidad de no incluir ninguna es aproximadamente $(1-0.001)^{32}\approx 96.8\%$. Por tanto, el Experience Replay queda dominado por tráfico normal y el agente recibe muy poca señal de aprendizaje sobre eventos críticos. Prioritized Experience Replay mitiga este problema al muestrear con mayor probabilidad las transiciones de mayor error TD: $P(i)\propto (|\delta_i|+\epsilon)^\alpha$. Las intrusiones, especialmente al inicio, suelen generar errores TD altos por ser inesperadas, por lo que se reutilizan más frecuentemente que bajo muestreo uniforme.

**d.** El sistema actual basado en reglas tiene una tasa de falsos positivos del 8%. Si diseñaran la función de recompensa de DQN para minimizar falsos positivos únicamente, ¿qué comportamiento indeseable podría aprender el agente? Diseñen una función de recompensa con al menos tres componentes que capturen el objetivo real del sistema, justificando la magnitud relativa de cada uno.

**Respuesta:**

Si la recompensa minimizara únicamente los falsos positivos, el agente podría aprender a ignorar la mayoría de los comportamientos sospechosos. Esto reduciría alertas incorrectas, pero incrementaría los falsos negativos al dejar intrusiones reales sin detectar ni contener.
Una función de recompensa debe balancear detección, contención y continuidad operativa:

$$
r =
40I_{\text{intrusión detectada}}
+80I_{\text{intrusión contenida}}
-200I_{\text{intrusión ignorada}}
-20I_{\text{acción defensiva sobre tráfico legítimo}}
-c(a)
$$

$$
c(\text{ignorar})=0,\quad
c(\text{alertar})=1,\quad
c(\text{bloquear})=5,\quad
c(\text{aislar})=15
$$

La penalización por ignorar una intrusión es la mayor porque un falso negativo puede causar daño grave. Detectarla tiene una recompensa positiva, y contenerla tiene una recompensa adicional para que el agente no se limite a alertar cuando bloquear o aislar sea necesario. Los falsos positivos se penalizan para mejorar la tasa actual del 8%, mientras que el costo creciente de las acciones evita bloqueos o aislamientos innecesarios.

### Task 1.2

El equipo propone usar Double DQN en lugar de DQN estándar.

**a.** Expliquen formalmente el problema de sobreestimación que Double DQN resuelve. En el contexto específico de detección de intrusiones, ¿qué consecuencia operacional tendría que el agente sobreestime sistemáticamente el valor de la acción "bloquear"? ¿Y de la acción "ignorar"?

**b.** Escriban la expresión del target de DQN estándar y la del target de Double DQN, identificando con precisión qué parámetros hacen qué en cada caso. ¿Cuál es el único cambio de implementación entre ambos?

**c.** Argumenten si en este dominio específico la sobreestimación es un problema más grave en ciertos tipos de estados que en otros. Consideren estados con alta ambigüedad (tráfico borderline) versus estados claramente maliciosos.

## Task 2

### Task 2.1


Implementen DQN y Double DQN sobre el entorno LunarLander-v2 de Gymnasium, que tiene espacio de acción discreto con 4 acciones y estado continuo de 8 dimensiones. Usen este entorno como proxy del sistema de detección de intrusiones para validar las propiedades algorítmicas antes de escalar al dominio real.

La red para $Q(s,a;w)$ debe tener dos capas ocultas de 128 neuronas con activación ReLU. El buffer de Experience Replay debe tener capacidad de 50,000 transiciones. El minibatch debe ser de 64 transiciones. La Target Network debe actualizarse cada 500 pasos. Usen $\epsilon$ decreciente de 1.0 a 0.05 durante los primeros 10,000 pasos.

Ambos algoritmos deben implementarse desde cero usando PyTorch. No se permite usar implementaciones preconstruidas de DQN de ninguna librería de RL.

La implementación debe registrar por episodio: recompensa total, valor $Q$ promedio estimado sobre un conjunto fijo de 100 estados de evaluación, y el target promedio usado en las actualizaciones. Estos tres registros son necesarios para detectar sobreestimación.

### Task 2.2


Con base en los resultados de la implementación, respondan:

**a.** Grafiquen la recompensa promedio por episodio de DQN y Double DQN en la misma figura con media móvil de 20 episodios. ¿Cuál converge más rápido? ¿Cuál alcanza mayor recompensa final? ¿El resultado coincide con lo esperado teóricamente?

**b.** Grafiquen el valor $Q$ promedio estimado durante el entrenamiento para ambos algoritmos. El valor $Q$ de DQN debería ser sistemáticamente mayor que el de Double DQN sobre los mismos estados. ¿Observan ese patrón? Si no lo observan, argumenten por qué podría no ser visible en este entorno específico.

**c.** Implementen ahora Prioritized Experience Replay sobre Double DQN, usando $\alpha = 0.6$ y $\beta$ creciente de 0.4 a 1.0. Comparen la velocidad de convergencia de Double DQN con muestreo uniforme versus Double DQN con PER. ¿En qué fase del entrenamiento es más visible la diferencia?

**d.** Argumenten si PER sería más o menos beneficioso en el dominio real de detección de intrusiones comparado con LunarLander, considerando la rareza de los eventos críticos que analizaron en la Task 1.1c. Apoyen su argumento con evidencia de sus experimentos.

### Task 2.3


Investigación bibliográfica y dictamen técnico.

**a.** Busquen y lean un paper publicado entre 2022 y 2025 que aplique DQN, Double DQN, Dueling DQN, PER, o Rainbow a un problema de ciberseguridad, detección de anomalías, o gestión de redes. El paper debe ser de IEEE Transactions on Information Forensics and Security, NeuroIPS, ICML, USENIX Security, o similar. Escriban un resumen técnico de media página con: el problema que resuelve, qué variante de DQN usa y por qué, los resultados principales, y una reflexión sobre si las limitaciones de DQN que identificaron en la Task 1 fueron resueltas en ese trabajo.

**b.** Redacten un dictamen técnico de un párrafo dirigido al equipo directivo de la empresa de ciberseguridad. El dictamen debe argumentar si recomiendan proceder con DQN o alguna de sus variantes para el sistema de detección de intrusiones, qué variante específica recomiendan y por qué, y qué pasos adicionales serían necesarios antes de un despliegue en producción. El dictamen debe estar respaldado por evidencia de sus experimentos, no solo por argumentos teóricos.